# Week 11 Lab: Shortest Paths and Minimum Spanning Trees

## Lab Overview

In this lab, you will apply **graph algorithms** (Dijkstra’s shortest path, Prim’s and Kruskal’s MST) to solve **practical, transport-network-inspired problems**.  
You will:

1. Construct and analyze transport graphs.
2. Implement path-finding using **Dijkstra’s algorithm**.
3. Apply **Prim’s and Kruskal’s algorithms** for MST construction.
4. Solve **real-world case-study inspired challenges** combining both shortest path and MST. 

## Question 1: Graph Construction for Transport Network

Design a **weighted, undirected graph class** representing a **city transport network**.  
- Each node represents a **station** (e.g., railway station, bus terminal, metro stop).  
- Each edge represents a **route** between two stations, with weights denoting **travel time in minutes**.  

Your graph class should support:
1. Adding/removing stations and routes.
2. Printing adjacency list and adjacency matrix representations.
3. Loading graph data from a `.csv` file containing `(station1, station2, travel_time)`.

In [ ]:
import csv

class TransportGraph:
    def __init__(self):
        self.graph = {}

    def add_station(self, station_name: str):
        if station_name not in self.graph:
            self.graph[station_name] = {}

    def add_route(self, station1: str, station2: str, travel_time: int):
        self.add_station(station1)
        self.add_station(station2)
        self.graph[station1][station2] = travel_time
        self.graph[station2][station1] = travel_time  

    def remove_station(self, station_name: str):
        if station_name in self.graph:
            for neighbor in list(self.graph[station_name].keys()):
                self.graph[neighbor].pop(station_name, None)
            self.graph.pop(station_name, None)

    def remove_route(self, station1: str, station2: str):
        if station1 in self.graph and station2 in self.graph[station1]:
            self.graph[station1].pop(station2, None)
        if station2 in self.graph and station1 in self.graph[station2]:
            self.graph[station2].pop(station1, None)

    def adjacency_list(self):
        return self.graph

    def adjacency_matrix(self):
        stations = list(self.graph.keys())
        size = len(stations)
        matrix = [[0] * size for _ in range(size)]

        for i, s1 in enumerate(stations):
            for j, s2 in enumerate(stations):
                if s2 in self.graph[s1]:
                    matrix[i][j] = self.graph[s1][s2]
        return stations, matrix

    def load_from_csv(self, filename: str):
        with open(filename, "r") as f:
            reader = csv.reader(f)
            for row in reader:
                station1, station2, travel_time = row[0], row[1], int(row[2])
                self.add_route(station1, station2, travel_time)

## Question 2: Shortest Path Planning (Dijkstra)

Implement Dijkstra’s Algorithm for the TransportGraph:

- Given a source station and a destination station, compute:
    - The shortest travel time.
    - The exact path of stations to follow.
- Ensure efficiency with a min-priority queue (heapq).
- Add functionality to compute shortest paths from a source to all other stations.

Then, solve the following practical tasks:

1. Find the shortest path between two busiest stations (to be chosen by you from dataset).
2. Compute all-station shortest paths from a central hub station.

In [ ]:
import heapq

class DijkstraSolver:
    def __init__(self, graph: TransportGraph):
        self.graph = graph.graph

    def shortest_path(self, source: str, destination: str):
        dist = {node: float("inf") for node in self.graph}
        dist[source] = 0

        parent = {node: None for node in self.graph}

        pq = [(0, source)]

        while pq:
            current_dist, node = heapq.heappop(pq)
            if node == destination:
                break

            if current_dist > dist[node]:
                continue

            for neighbor, weight in self.graph[node].items():
                new_dist = current_dist + weight
                if new_dist < dist[neighbor]:
                    dist[neighbor] = new_dist
                    parent[neighbor] = node
                    heapq.heappush(pq, (new_dist, neighbor))

        path = []
        cur = destination
        while cur is not None:
            path.append(cur)
            cur = parent[cur]
        path.reverse()

        return dist[destination], path

    def all_shortest_paths(self, source: str):
        dist = {node: float("inf") for node in self.graph}
        dist[source] = 0
        parent = {node: None for node in self.graph}

        pq = [(0, source)]
        while pq:
            current_dist, node = heapq.heappop(pq)

            if current_dist > dist[node]:
                continue

            for neighbor, weight in self.graph[node].items():
                new_dist = current_dist + weight
                if new_dist < dist[neighbor]:
                    dist[neighbor] = new_dist
                    parent[neighbor] = node
                    heapq.heappush(pq, (new_dist, neighbor))

        paths = {}
        for dest in self.graph:
            if dist[dest] == float("inf"):
                paths[dest] = (float("inf"), [])
            else:
                path = []
                cur = dest
                while cur is not None:
                    path.append(cur)
                    cur = parent[cur]
                path.reverse()
                paths[dest] = (dist[dest], path)

        return paths

## Question 3: Minimum Spanning Tree (Prim & Kruskal)

Extend the transport network by constructing a Minimum Spanning Tree (MST):

- Implement Prim’s algorithm.
- Implement Kruskal’s algorithm (use Union-Find/Disjoint Set Union).
- Compare results of both algorithms:
    - Total cost (sum of weights).
    - Edge set in the MST.

Then, answer the following practical challenge:

- Imagine the city is trying to lay down fiber-optic cables connecting all stations with minimum cost.
- Which algorithm (Prim or Kruskal) is better suited in practice for this case, and why?

In [ ]:
class DisjointSet:
    def __init__(self, vertices):
        self.parent = {v: v for v in vertices}
        self.rank = {v: 0 for v in vertices}

    def find(self, item):
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])  
        return self.parent[item]

    def union(self, set1, set2):
        root1, root2 = self.find(set1), self.find(set2)
        if root1 == root2:
            return False
        if self.rank[root1] < self.rank[root2]:
            self.parent[root1] = root2
        elif self.rank[root1] > self.rank[root2]:
            self.parent[root2] = root1
        else:
            self.parent[root2] = root1
            self.rank[root1] += 1
        return True


class MSTSolver:
    def __init__(self, graph: TransportGraph):
        self.graph = graph.graph

    def prim_mst(self):
        start = next(iter(self.graph))  
        visited = set([start])
        edges = []
        mst_edges = []
        total_cost = 0

        for neighbor, weight in self.graph[start].items():
            heapq.heappush(edges, (weight, start, neighbor))

        while edges and len(visited) < len(self.graph):
            weight, u, v = heapq.heappop(edges)
            if v not in visited:
                visited.add(v)
                mst_edges.append((u, v, weight))
                total_cost += weight
                for neighbor, w in self.graph[v].items():
                    if neighbor not in visited:
                        heapq.heappush(edges, (w, v, neighbor))

        return total_cost, mst_edges

    def kruskal_mst(self):
        all_edges = []
        for u in self.graph:
            for v, w in self.graph[u].items():
                if u < v:  
                    all_edges.append((w, u, v))

        all_edges.sort()
        ds = DisjointSet(self.graph.keys())
        mst_edges = []
        total_cost = 0

        for w, u, v in all_edges:
            if ds.union(u, v):
                mst_edges.append((u, v, w))
                total_cost += w

        return total_cost, mst_edges